  # 05 — Statistical Significance Validation of Spectral and FPLS Models



  This notebook reruns selected models on the same repeated stratified cross-validation folds and evaluates whether observed performance differences are statistically significant.



  Compared model groups:



  A. Models trained on the same FPLS representation:



  - FPLS + linear SVM with J = 5;

  - FPLS + logistic regression with J = 5;

  - FPLS + weighted kNN with J = 5.



  B. Direct spectra vs FPLS representation:



  - direct linear SVM vs FPLS + linear SVM;

  - direct logistic regression vs FPLS + logistic regression;

  - direct weighted kNN SEUC vs FPLS + weighted kNN.



  C. Direct spectral models:



  - direct logistic regression vs direct linear SVM;

  - direct logistic regression vs direct weighted kNN SEUC.



  Main comparison metrics:



  - F1;

  - PR-AUC.



  Main statistical test:



  - paired Wilcoxon signed-rank test.



  Outputs:



  - fold-level model metrics;

  - summary table by method;

  - pairwise Wilcoxon test table;

  - F1 and PR-AUC boxplots.

In [1]:
from __future__ import annotations

import json
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import wilcoxon
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")




  ## 1. General configuration







  This section defines the main experiment settings.







  - `RANDOM_STATE` keeps model behaviour reproducible.



  - `SMOKE` can be set to `True` to test only the first repeated CV block.



  - `FPLS_J` defines the number of FPLS components.



  - `K_KNN` defines the number of neighbours used by weighted kNN.



  - `PAIRWISE_COMPARISONS` defines the method pairs tested with Wilcoxon tests.

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SMOKE = False

FPLS_J = 5
K_KNN = 5

PAIRWISE_COMPARISONS = [
    # A. Comparisons inside the same FPLS representation
    {
        "group": "A. FPLS model comparison",
        "method_a": "FPLS + linear SVM (J=5)",
        "method_b": "FPLS + logistic regression (J=5)",
    },
    {
        "group": "A. FPLS model comparison",
        "method_a": "FPLS + linear SVM (J=5)",
        "method_b": "FPLS + weighted kNN (J=5)",
    },
    {
        "group": "A. FPLS model comparison",
        "method_a": "FPLS + logistic regression (J=5)",
        "method_b": "FPLS + weighted kNN (J=5)",
    },

    # B. Direct spectra vs FPLS representation
    {
        "group": "B. Direct spectra vs FPLS",
        "method_a": "Direct linear SVM",
        "method_b": "FPLS + linear SVM (J=5)",
    },
    {
        "group": "B. Direct spectra vs FPLS",
        "method_a": "Direct logistic regression (L2)",
        "method_b": "FPLS + logistic regression (J=5)",
    },
    {
        "group": "B. Direct spectra vs FPLS",
        "method_a": "Direct weighted kNN + standardized Euclidean",
        "method_b": "FPLS + weighted kNN (J=5)",
    },

    # C. Direct spectral model comparison
    {
        "group": "C. Direct spectral model comparison",
        "method_a": "Direct logistic regression (L2)",
        "method_b": "Direct linear SVM",
    },
    {
        "group": "C. Direct spectral model comparison",
        "method_a": "Direct logistic regression (L2)",
        "method_b": "Direct weighted kNN + standardized Euclidean",
    },
]




  ## 2. Paths and input files







  The notebook expects the input files to be stored in `og_data`.







  Required files:







  - `xp_sampled_spectra.csv`: sampled Gaia XP spectra;



  - `splits_rskf.json`: predefined repeated stratified CV folds.







  Optional file:







  - `og_xp.csv`: label file used only if labels are not already included in



    `xp_sampled_spectra.csv`.







  Outputs are saved to `results/05_significance_validation`.

In [3]:
BASE_DIR = Path.cwd() / "og_data"
OUT_DIR = Path.cwd() / "results" / "05_significance_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_FILE = BASE_DIR / "splits_rskf.json"

SAMPLED_CANDIDATES = [
    BASE_DIR / "xp_sampled_spectra.csv",
]

LABEL_CANDIDATES = [
    BASE_DIR / "og_xp.csv",
]




  ## 3. Plot style




  This section defines a consistent visual style.

In [4]:
COLOR_PRIMARY = "#6193CD"
COLOR_SECONDARY = "#8C8C8C"
COLOR_DARK = "#4A4A4A"
COLOR_LIGHT = "#D9D9D9"
COLOR_TEXT = "#222222"

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 11,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "axes.edgecolor": "#444444",
        "axes.linewidth": 0.8,
        "axes.labelcolor": COLOR_TEXT,
        "xtick.color": COLOR_TEXT,
        "ytick.color": COLOR_TEXT,
        "text.color": COLOR_TEXT,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    }
)




  ## 4. Helper functions

  These functions handle plotting, file lookup, score normalization, thresholding, metric calculation, and summary formatting.

In [ ]:
def apply_clean_axes(
    ax,
    add_grid: bool = False,
    grid_axis: str = "y",
) -> None:
    """Apply minimal formatting to a matplotlib axis.

    Parameters
    ----------
    ax:
        Matplotlib axis object.
    add_grid:
        Whether to add a light dashed grid.
    grid_axis:
        Axis along which the grid should be drawn.
    """
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if add_grid:
        ax.grid(
            True,
            axis=grid_axis,
            linestyle="--",
            linewidth=0.6,
            alpha=0.5,
            color=COLOR_LIGHT,
        )
    else:
        ax.grid(False)


def show_and_save(fig, path: Path) -> None:
    """Show a figure and immediately save it as an SVG file.

    This keeps each figure export next to its plotting block and removes the
    need for a separate figure-saving section.

    Parameters
    ----------
    fig:
        Matplotlib figure object.
    path:
        Output SVG path.
    """
    plt.tight_layout()
    plt.show()
    fig.savefig(path, format="svg", bbox_inches="tight", facecolor="white")
    plt.close(fig)


def find_first_existing(paths: Sequence[Path]) -> Path | None:
    """Return the first existing path from a list of candidate paths.

    Parameters
    ----------
    paths:
        Candidate file paths.

    Returns
    -------
    Path | None
        First existing path, or `None` if no file exists.
    """
    for path in paths:
        path = Path(path)

        if path.exists():
            return path

    return None


def split_sort_key(split_name: str) -> tuple[int, int]:
    """Sort repeated CV split names by repetition and fold number.

    The expected split name format is similar to `rep0_fold0`.

    Parameters
    ----------
    split_name:
        Split name from the JSON split dictionary.

    Returns
    -------
    tuple[int, int]
        Parsed repetition and fold numbers.
    """
    rep = int(split_name.split("_")[0].replace("rep", ""))
    fold = int(split_name.split("_")[1].replace("fold", ""))

    return rep, fold


def mean_std_string(mean: float, std: float) -> str | float:
    """Format a metric as `mean ± std`.

    Parameters
    ----------
    mean:
        Mean metric value.
    std:
        Standard deviation of the metric.

    Returns
    -------
    str | float
        Formatted metric string, or `np.nan` if the mean is missing.
    """
    if pd.isna(mean):
        return np.nan

    std_value = 0.0 if pd.isna(std) else std
    return f"{mean:.4f} ± {std_value:.4f}"


def normalize_scores_train_ref(
    scores_te: np.ndarray,
    scores_tr: np.ndarray,
) -> np.ndarray:
    """Normalize scores using the training-score range.

    Test scores are normalized using only the training fold minimum and maximum.
    This avoids leaking test-fold information into threshold selection.

    Parameters
    ----------
    scores_te:
        Scores to normalize.
    scores_tr:
        Training scores used as the reference range.

    Returns
    -------
    np.ndarray
        Scores scaled and clipped to `[0, 1]`.
    """
    lo = float(np.min(scores_tr))
    hi = float(np.max(scores_tr))

    if hi == lo:
        return np.full_like(scores_te, 0.5, dtype=np.float64)

    normalized = ((scores_te - lo) / (hi - lo)).astype(np.float64)

    return np.clip(normalized, 0.0, 1.0)


def pick_youden_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    grid_size: int = 200,
) -> float:
    """Choose a classification threshold by maximizing Youden's J.

    Youden's J is defined as sensitivity + specificity - 1. The threshold is
    selected on training-fold scores and later applied to the test fold.

    Parameters
    ----------
    y_true:
        True binary labels.
    y_prob:
        Normalized class-1 scores.
    grid_size:
        Number of candidate thresholds between 0 and 1.

    Returns
    -------
    float
        Threshold with the highest Youden's J statistic.
    """
    thresholds = np.linspace(0, 1, grid_size)
    best_j = -1.0
    best_threshold = 0.5

    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(np.int64)

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
        specificity = tn / (tn + fp) if (tn + fp) else 0.0
        youden_j = sensitivity + specificity - 1.0

        if youden_j > best_j:
            best_j = youden_j
            best_threshold = float(threshold)

    return best_threshold


def fold_metrics(
    y_true_te: np.ndarray,
    y_score_te: np.ndarray,
    y_true_tr: np.ndarray,
    y_score_tr: np.ndarray,
) -> dict[str, float]:
    """Calculate fold-level classification metrics.

    PR-AUC and ROC-AUC are calculated from continuous scores. Binary metrics
    are calculated after choosing a Youden threshold on the training fold.

    Parameters
    ----------
    y_true_te:
        True labels for the test fold.
    y_score_te:
        Continuous scores for the test fold.
    y_true_tr:
        True labels for the training fold.
    y_score_tr:
        Continuous scores for the training fold.

    Returns
    -------
    dict[str, float]
        Fold-level metrics.
    """
    metrics = {
        "pr_auc": average_precision_score(y_true_te, y_score_te),
    }

    try:
        metrics["roc_auc"] = float(roc_auc_score(y_true_te, y_score_te))
    except ValueError:
        metrics["roc_auc"] = np.nan

    prob_tr = normalize_scores_train_ref(y_score_tr, y_score_tr)
    prob_te = normalize_scores_train_ref(y_score_te, y_score_tr)

    threshold = pick_youden_threshold(y_true_tr, prob_tr)
    y_pred = (prob_te >= threshold).astype(np.int64)

    metrics["youden_threshold"] = threshold
    metrics["sensitivity"] = recall_score(
        y_true_te,
        y_pred,
        pos_label=1,
        zero_division=0,
    )
    metrics["precision"] = precision_score(
        y_true_te,
        y_pred,
        pos_label=1,
        zero_division=0,
    )
    metrics["specificity"] = recall_score(
        y_true_te,
        y_pred,
        pos_label=0,
        zero_division=0,
    )
    metrics["accuracy"] = accuracy_score(y_true_te, y_pred)
    metrics["f1"] = f1_score(
        y_true_te,
        y_pred,
        pos_label=1,
        zero_division=0,
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true_te,
        y_pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    metrics["youden_j"] = sensitivity + specificity - 1.0

    return metrics


def summarise_run(
    df_run: pd.DataFrame,
    group_cols: list[str],
) -> pd.DataFrame:
    """Summarise fold-level results by selected grouping columns.

    Parameters
    ----------
    df_run:
        Fold-level metric table.
    group_cols:
        Columns used for grouping the results.

    Returns
    -------
    pd.DataFrame
        Summary table containing metric means and standard deviations.
    """
    metric_cols = [
        "pr_auc",
        "roc_auc",
        "sensitivity",
        "precision",
        "specificity",
        "accuracy",
        "f1",
        "youden_j",
        "youden_threshold",
    ]

    agg_dict = {}

    for metric in metric_cols:
        agg_dict[f"{metric}_mean"] = pd.NamedAgg(
            column=metric,
            aggfunc="mean",
        )
        agg_dict[f"{metric}_std"] = pd.NamedAgg(
            column=metric,
            aggfunc="std",
        )

    return df_run.groupby(group_cols).agg(**agg_dict).reset_index()




  ## 5. Load sampled spectra, labels, and saved folds


  The sampled spectra file is used as the main source. If it already contains `y`, labels are loaded directly from it. Otherwise, labels are merged from `og_xp.csv`.

In [6]:
SAMPLED_FILE = find_first_existing(SAMPLED_CANDIDATES)

if SAMPLED_FILE is None:
    raise FileNotFoundError("Could not find sampled spectra file.")

if not SPLIT_FILE.exists():
    raise FileNotFoundError(f"Missing split file: {SPLIT_FILE}")

df_sampled = pd.read_csv(SAMPLED_FILE)

if "source_id" not in df_sampled.columns:
    raise ValueError("Sampled spectra file must contain `source_id`.")

wl_cols = [col for col in df_sampled.columns if str(col).startswith("wl_")]

if len(wl_cols) == 0:
    raise ValueError("Sampled spectra file must contain `wl_*` columns.")

if "y" in df_sampled.columns:
    df_m = df_sampled[["source_id", "y"] + wl_cols].copy()
else:
    LABEL_FILE = find_first_existing(LABEL_CANDIDATES)

    if LABEL_FILE is None:
        raise FileNotFoundError(
            "Could not find label file and sampled file has no `y` column."
        )

    df_labels = pd.read_csv(LABEL_FILE)

    if "source_id" not in df_labels.columns or "y" not in df_labels.columns:
        raise ValueError("Label file must contain `source_id` and `y`.")

    df_m = df_labels[["source_id", "y"]].merge(
        df_sampled[["source_id"] + wl_cols],
        on="source_id",
        how="inner",
        validate="one_to_one",
    )

with open(SPLIT_FILE, encoding="utf-8") as file:
    splits = json.load(file)

split_names = sorted(splits.keys(), key=split_sort_key)

if SMOKE:
    split_names = [
        split_name
        for split_name in split_names
        if split_name.startswith("rep0_")
    ]

F_raw = df_m[wl_cols].to_numpy(dtype=np.float64)
y_all = df_m["y"].to_numpy(dtype=np.int64)

# Apply row-wise L2 normalisation to focus on spectral shape.
norms = np.linalg.norm(F_raw, axis=1, keepdims=True)
norms = np.maximum(norms, 1e-15)
F_sampled = F_raw / norms

print("Using sampled file:", SAMPLED_FILE.name)
print("Matrix shape:", F_sampled.shape)
print("Number of splits:", len(split_names))
print("Class counts:")
print(pd.Series(y_all).value_counts().sort_index())




Using sampled file: xp_sampled_spectra.csv
Matrix shape: (2815, 343)
Number of splits: 50
Class counts:
0    2257
1     558
Name: count, dtype: int64


  ## 6. Model definitions


  `ModelSpec` stores the display label and the internal builder name for each model used in the significance comparison.

In [ ]:
@dataclass
class ModelSpec:
    """Specification of one model used in significance testing.

    Attributes
    ----------
    method_label:
        Human-readable method name used in result tables and plots.
    builder_name:
        Internal identifier used to select the model runner function.
    representation:
        Input representation used by the model.
    """

    method_label: str
    builder_name: str
    representation: str


MODEL_SPECS = [
    ModelSpec(
        method_label="Direct linear SVM",
        builder_name="direct_svm",
        representation="Direct L2-normalized spectra",
    ),
    ModelSpec(
        method_label="Direct logistic regression (L2)",
        builder_name="direct_lr",
        representation="Direct L2-normalized spectra",
    ),
    ModelSpec(
        method_label="Direct weighted kNN + standardized Euclidean",
        builder_name="direct_knn_seuclidean",
        representation="Direct L2-normalized spectra",
    ),
    ModelSpec(
        method_label="FPLS + linear SVM (J=5)",
        builder_name="fpls_svm",
        representation="FPLS scores",
    ),
    ModelSpec(
        method_label="FPLS + logistic regression (J=5)",
        builder_name="fpls_lr",
        representation="FPLS scores",
    ),
    ModelSpec(
        method_label="FPLS + weighted kNN (J=5)",
        builder_name="fpls_knn",
        representation="FPLS scores",
    ),
]

METHOD_ORDER = [spec.method_label for spec in MODEL_SPECS]

  ## 7. Model-specific helper functions


  These functions implement each compared model and return continuous scores for the training and test folds.

In [8]:
def fpls_transform(
    F_tr: np.ndarray,
    y_tr: np.ndarray,
    F_te: np.ndarray,
    n_components: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Fit FPLS on the training fold and transform train and test spectra.

    Training spectra are mean-centered first. The same training mean is used to
    center the test spectra.

    Parameters
    ----------
    F_tr:
        Training spectra.
    y_tr:
        Training labels.
    F_te:
        Test spectra.
    n_components:
        Requested number of FPLS components.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Training and test FPLS scores.
    """
    train_mean = F_tr.mean(axis=0)
    F_tr_centered = F_tr - train_mean
    F_te_centered = F_te - train_mean

    max_allowed_j = min(
        F_tr_centered.shape[0] - 1,
        F_tr_centered.shape[1],
        n_components,
    )
    max_allowed_j = max(max_allowed_j, 1)

    pls = PLSRegression(n_components=max_allowed_j)
    pls.fit(F_tr_centered, y_tr)

    xi_tr = pls.transform(F_tr_centered)
    xi_te = pls.transform(F_te_centered)

    return xi_tr, xi_te


def weighted_knn_scores(
    distances: np.ndarray,
    y_ref: np.ndarray,
    k: int = 5,
) -> np.ndarray:
    """Convert distances to inverse-distance weighted kNN scores.

    Parameters
    ----------
    distances:
        Pairwise distance matrix between query and reference objects.
    y_ref:
        Labels of reference objects.
    k:
        Number of nearest neighbours.

    Returns
    -------
    np.ndarray
        Continuous class-1 scores.
    """
    neighbour_idx = np.argsort(distances, axis=1)[:, :k]
    neighbour_distances = np.take_along_axis(
        distances,
        neighbour_idx,
        axis=1,
    )
    neighbour_labels = y_ref[neighbour_idx]

    weights = 1.0 / np.maximum(neighbour_distances, 1e-12)
    scores = (
        weights * neighbour_labels
    ).sum(axis=1) / weights.sum(axis=1)

    return scores
def run_direct_svm(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_te: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Run a linear SVM directly on L2-normalized spectra.

    Parameters
    ----------
    X_tr:
        Training spectra.
    y_tr:
        Training labels.
    X_te:
        Test spectra.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Training and test decision scores.
    """
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)

    clf = SVC(
        C=1.0,
        kernel="linear",
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )
    clf.fit(X_tr_scaled, y_tr)

    score_tr = clf.decision_function(X_tr_scaled)
    score_te = clf.decision_function(X_te_scaled)

    return score_tr, score_te


def run_direct_lr(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_te: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Run L2-regularized logistic regression directly on spectra.

    Parameters
    ----------
    X_tr:
        Training spectra.
    y_tr:
        Training labels.
    X_te:
        Test spectra.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Training and test decision scores.
    """
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)

    clf = LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=5000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )
    clf.fit(X_tr_scaled, y_tr)

    score_tr = clf.decision_function(X_tr_scaled)
    score_te = clf.decision_function(X_te_scaled)

    return score_tr, score_te

def run_fpls_svm(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_te: np.ndarray,
    n_components: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    """Run FPLS followed by a linear SVM.

    Parameters
    ----------
    X_tr:
        Training spectra.
    y_tr:
        Training labels.
    X_te:
        Test spectra.
    n_components:
        Number of FPLS components.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Training and test decision scores.
    """
    xi_tr, xi_te = fpls_transform(X_tr, y_tr, X_te, n_components)

    scaler = StandardScaler()
    xi_tr_scaled = scaler.fit_transform(xi_tr)
    xi_te_scaled = scaler.transform(xi_te)

    clf = SVC(
        C=1.0,
        kernel="linear",
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )
    clf.fit(xi_tr_scaled, y_tr)

    score_tr = clf.decision_function(xi_tr_scaled)
    score_te = clf.decision_function(xi_te_scaled)

    return score_tr, score_te


def run_fpls_lr(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_te: np.ndarray,
    n_components: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    """Run FPLS followed by logistic regression.

    Parameters
    ----------
    X_tr:
        Training spectra.
    y_tr:
        Training labels.
    X_te:
        Test spectra.
    n_components:
        Number of FPLS components.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Training and test decision scores.
    """
    xi_tr, xi_te = fpls_transform(X_tr, y_tr, X_te, n_components)

    scaler = StandardScaler()
    xi_tr_scaled = scaler.fit_transform(xi_tr)
    xi_te_scaled = scaler.transform(xi_te)

    clf = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )
    clf.fit(xi_tr_scaled, y_tr)

    score_tr = clf.decision_function(xi_tr_scaled)
    score_te = clf.decision_function(xi_te_scaled)

    return score_tr, score_te


def run_knn_seuclidean(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_te: np.ndarray,
    k: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    """Run weighted kNN with standardized Euclidean distance.

    Standardisation and variance estimation are fitted only on the training
    fold.

    Parameters
    ----------
    X_tr:
        Training spectra.
    y_tr:
        Training labels.
    X_te:
        Test spectra.
    k:
        Number of nearest neighbours.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Training and test weighted kNN scores.
    """
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)

    var_tr = X_tr_scaled.var(axis=0, ddof=1)
    var_tr = np.where(var_tr < 1e-12, 1e-12, var_tr)

    dists_tr = pairwise_distances(
        X_tr_scaled,
        X_tr_scaled,
        metric="seuclidean",
        V=var_tr,
    )
    dists_te = pairwise_distances(
        X_te_scaled,
        X_tr_scaled,
        metric="seuclidean",
        V=var_tr,
    )

    # Exclude each training object from its own neighbour list.
    np.fill_diagonal(dists_tr, np.inf)

    score_tr = weighted_knn_scores(dists_tr, y_tr, k=k)
    score_te = weighted_knn_scores(dists_te, y_tr, k=k)

    return score_tr, score_te


def run_fpls_knn(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_te: np.ndarray,
    n_components: int = 5,
    k: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    """Run FPLS followed by weighted kNN.

    Parameters
    ----------
    X_tr:
        Training spectra.
    y_tr:
        Training labels.
    X_te:
        Test spectra.
    n_components:
        Number of FPLS components.
    k:
        Number of nearest neighbours.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Training and test weighted kNN scores.
    """
    xi_tr, xi_te = fpls_transform(X_tr, y_tr, X_te, n_components)

    scaler = StandardScaler()
    xi_tr_scaled = scaler.fit_transform(xi_tr)
    xi_te_scaled = scaler.transform(xi_te)

    dists_tr = pairwise_distances(
        xi_tr_scaled,
        xi_tr_scaled,
        metric="euclidean",
    )
    dists_te = pairwise_distances(
        xi_te_scaled,
        xi_tr_scaled,
        metric="euclidean",
    )

    # Exclude each training object from its own neighbour list.
    np.fill_diagonal(dists_tr, np.inf)

    score_tr = weighted_knn_scores(dists_tr, y_tr, k=k)
    score_te = weighted_knn_scores(dists_te, y_tr, k=k)

    return score_tr, score_te




  ## 8. Unified fold runner


  This function runs one selected model on all predefined CV folds and returns fold-level metrics.

In [9]:
def run_model_on_all_folds(spec: ModelSpec) -> pd.DataFrame:
    """Run one model specification on all repeated CV folds.

    Parameters
    ----------
    spec:
        Model specification containing method label, builder name, and
        representation.

    Returns
    -------
    pd.DataFrame
        Fold-level metrics for one method.
    """
    records = []

    for split_name in split_names:
        train_idx = np.array(splits[split_name]["train"], dtype=int)
        test_idx = np.array(splits[split_name]["test"], dtype=int)

        X_tr = F_sampled[train_idx]
        X_te = F_sampled[test_idx]
        y_tr = y_all[train_idx]
        y_te = y_all[test_idx]

        if spec.builder_name == "direct_svm":
            score_tr, score_te = run_direct_svm(
                X_tr,
                y_tr,
                X_te,
            )

        elif spec.builder_name == "direct_lr":
            score_tr, score_te = run_direct_lr(
                X_tr,
                y_tr,
                X_te,
            )

        elif spec.builder_name == "direct_knn_seuclidean":
            score_tr, score_te = run_knn_seuclidean(
                X_tr,
                y_tr,
                X_te,
                k=K_KNN,
            )

        elif spec.builder_name == "fpls_svm":
            score_tr, score_te = run_fpls_svm(
                X_tr,
                y_tr,
                X_te,
                n_components=FPLS_J,
            )

        elif spec.builder_name == "fpls_lr":
            score_tr, score_te = run_fpls_lr(
                X_tr,
                y_tr,
                X_te,
                n_components=FPLS_J,
            )

        elif spec.builder_name == "fpls_knn":
            score_tr, score_te = run_fpls_knn(
                X_tr,
                y_tr,
                X_te,
                n_components=FPLS_J,
                k=K_KNN,
            )

        else:
            raise ValueError(f"Unknown builder: {spec.builder_name}")

        metrics = fold_metrics(y_te, score_te, y_tr, score_tr)

        records.append(
            {
                "split": split_name,
                "method": spec.method_label,
                "representation": spec.representation,
                **metrics,
            }
        )

    return pd.DataFrame(records)




  ## 9. Run all selected models

 Each model is rerun on the same fold sequence. This creates paired metric values that can be compared with the Wilcoxon signed-rank test.

In [10]:
all_runs = []

for spec in MODEL_SPECS:
    print(f"Running: {spec.method_label}")

    df_model = run_model_on_all_folds(spec)
    all_runs.append(df_model)

df_fold = pd.concat(all_runs, ignore_index=True)

print("\nFold-level results preview:")
display(df_fold.head())




Running: Direct linear SVM
Running: Direct logistic regression (L2)
Running: Direct weighted kNN + standardized Euclidean
Running: FPLS + linear SVM (J=5)
Running: FPLS + logistic regression (J=5)
Running: FPLS + weighted kNN (J=5)

Fold-level results preview:


,split,method,representation,pr_auc,roc_auc,youden_threshold,sensitivity,precision,specificity,accuracy,f1,youden_j
0,rep0_fold0,Direct linear SVM,Direct L2-normalized spectra,0.927447,0.955712,0.206030,0.873874,0.850877,0.962389,0.944938,0.862222,0.836263
1,rep0_fold1,Direct linear SVM,Direct L2-normalized spectra,0.818025,0.899127,0.155779,0.774775,0.741379,0.933628,0.902309,0.757709,0.708403
2,rep0_fold2,Direct linear SVM,Direct L2-normalized spectra,0.857979,0.932076,0.150754,0.821429,0.779661,0.942350,0.918295,0.800000,0.763779
3,rep0_fold3,Direct linear SVM,Direct L2-normalized spectra,0.898070,0.950071,0.160804,0.830357,0.845455,0.962306,0.936057,0.837838,0.792663
4,rep0_fold4,Direct linear SVM,Direct L2-normalized spectra,0.839499,0.934530,0.140704,0.839286,0.789916,0.944568,0.923623,0.813853,0.783853


  ## 10. Summary table







  Fold-level metrics are summarised by method.

In [11]:
df_summary = summarise_run(df_fold, ["representation", "method"])

df_summary["method"] = pd.Categorical(
    df_summary["method"],
    categories=METHOD_ORDER,
    ordered=True,
)

df_summary = df_summary.sort_values("method").reset_index(drop=True)

print("\n=== SUMMARY ===")
display(
    df_summary[
        [
            "representation",
            "method",
            "f1_mean",
            "pr_auc_mean",
            "roc_auc_mean",
            "youden_j_mean",
        ]
    ]
)





=== SUMMARY ===


,representation,method,f1_mean,pr_auc_mean,roc_auc_mean,youden_j_mean
0,Direct L2-normalized spectra,Direct linear SVM,0.812075,0.866675,0.933266,0.773875
1,Direct L2-normalized spectra,Direct logistic regression (L2),0.805179,0.868510,0.934864,0.772695
2,Direct L2-normalized spectra,Direct weighted kNN + standardized Euclidean,0.682221,0.739134,0.877118,0.635985
3,FPLS scores,FPLS + linear SVM (J=5),0.808243,0.865758,0.930481,0.774222
4,FPLS scores,FPLS + logistic regression (J=5),0.807004,0.869791,0.931930,0.774651
5,FPLS scores,FPLS + weighted kNN (J=5),0.800105,0.821988,0.897409,0.729855


  ## 11. Paired Wilcoxon tests

  The Wilcoxon signed-rank test is applied to paired fold-level metric values. For each method pair, F1 and PR-AUC are tested separately.

In [12]:
wilcoxon_rows = []

for comparison in PAIRWISE_COMPARISONS:
    group = comparison["group"]
    method_a = comparison["method_a"]
    method_b = comparison["method_b"]

    df_a = df_fold[df_fold["method"] == method_a].sort_values("split")
    df_b = df_fold[df_fold["method"] == method_b].sort_values("split")

    common_splits = sorted(
        set(df_a["split"]).intersection(set(df_b["split"]))
    )

    df_a = df_a[df_a["split"].isin(common_splits)].sort_values("split")
    df_b = df_b[df_b["split"].isin(common_splits)].sort_values("split")

    if len(df_a) == 0 or len(df_b) == 0:
        continue

    for metric in ["f1", "pr_auc"]:
        values_a = df_a[metric].to_numpy(dtype=float)
        values_b = df_b[metric].to_numpy(dtype=float)

        try:
            stat, p_value = wilcoxon(
                values_a,
                values_b,
                zero_method="wilcox",
                alternative="two-sided",
            )
        except ValueError:
            stat = np.nan
            p_value = np.nan

        wilcoxon_rows.append(
            {
                "comparison_group": group,
                "method_a": method_a,
                "method_b": method_b,
                "metric": metric,
                "n_pairs": len(values_a),
                "mean_a": np.mean(values_a),
                "mean_b": np.mean(values_b),
                "mean_diff_a_minus_b": np.mean(values_a - values_b),
                "wilcoxon_stat": stat,
                "p_value": p_value,
                "significant_0_05": (
                    bool(p_value < 0.05)
                    if not pd.isna(p_value)
                    else False
                ),
            }
        )

df_wilcoxon = pd.DataFrame(wilcoxon_rows)

print("\n=== WILCOXON RESULTS ===")
display(df_wilcoxon)





=== WILCOXON RESULTS ===


,comparison_group,method_a,method_b,metric,n_pairs,mean_a,mean_b,mean_diff_a_minus_b,wilcoxon_stat,p_value,significant_0_05
0,A. FPLS model comparison,FPLS + linear SVM (J=5),FPLS + logistic regression (J=5),f1,50,0.808243,0.807004,0.001239,606.0,7.667327e-01,False
1,A. FPLS model comparison,FPLS + linear SVM (J=5),FPLS + logistic regression (J=5),pr_auc,50,0.865758,0.869791,-0.004032,3.0,8.881784e-15,True
2,A. FPLS model comparison,FPLS + linear SVM (J=5),FPLS + weighted kNN (J=5),f1,50,0.808243,0.800105,0.008138,434.0,4.944641e-02,True
3,A. FPLS model comparison,FPLS + linear SVM (J=5),FPLS + weighted kNN (J=5),pr_auc,50,0.865758,0.821988,0.043770,0.0,1.776357e-15,True
4,A. FPLS model comparison,FPLS + logistic regression (J=5),FPLS + weighted kNN (J=5),f1,50,0.807004,0.800105,0.006899,481.0,1.329606e-01,False
5,A. FPLS model comparison,FPLS + logistic regression (J=5),FPLS + weighted kNN (J=5),pr_auc,50,0.869791,0.821988,0.047802,0.0,1.776357e-15,True
6,B. Direct spectra vs FPLS,Direct linear SVM,FPLS + linear SVM (J=5),f1,50,0.812075,0.808243,0.003832,531.0,3.090510e-01,False
7,B. Direct spectra vs FPLS,Direct linear SVM,FPLS + linear SVM (J=5),pr_auc,50,0.866675,0.865758,0.000917,524.0,2.779880e-01,False
8,B. Direct spectra vs FPLS,Direct logistic regression (L2),FPLS + logistic regression (J=5),f1,50,0.805179,0.807004,-0.001825,579.0,7.389574e-01,False
9,B. Direct spectra vs FPLS,Direct logistic regression (L2),FPLS + logistic regression (J=5),pr_auc,50,0.868510,0.869791,-0.001281,540.0,3.521750e-01,False


  ## 12. Save table outputs

This section saves the CSV result tables.

In [15]:
fold_path = OUT_DIR / "significance_fold_metrics.csv"
summary_path = OUT_DIR / "significance_summary.csv"
wilcoxon_path = OUT_DIR / "significance_wilcoxon.csv"

df_fold.to_csv(fold_path, index=False)
df_summary.to_csv(summary_path, index=False)
df_wilcoxon.to_csv(wilcoxon_path, index=False)

print("Saved main CSV outputs to:", OUT_DIR)




Saved main CSV outputs to: c:\Users\Lenovo\Documents\VDA\Bakalauras\Kodas\results\05_significance_validation


  ## 113. Final quick view

  This final display shows compact mean ± standard deviation summaries and the Wilcoxon test results.

In [16]:
df_summary_display = df_summary.copy()

pretty_summary = pd.DataFrame(
    {
        "Representation": df_summary_display["representation"],
        "Method": df_summary_display["method"].astype(str),
        "F1": [
            mean_std_string(mean, std)
            for mean, std in zip(
                df_summary_display["f1_mean"],
                df_summary_display["f1_std"],
            )
        ],
        "PR-AUC": [
            mean_std_string(mean, std)
            for mean, std in zip(
                df_summary_display["pr_auc_mean"],
                df_summary_display["pr_auc_std"],
            )
        ],
        "ROC-AUC": [
            mean_std_string(mean, std)
            for mean, std in zip(
                df_summary_display["roc_auc_mean"],
                df_summary_display["roc_auc_std"],
            )
        ],
    }
)

display(pretty_summary)
display(df_wilcoxon)




,Representation,Method,F1,PR-AUC,ROC-AUC
0,Direct L2-normalized spectra,Direct linear SVM,0.8121 ± 0.0279,0.8667 ± 0.0309,0.9333 ± 0.0142
1,Direct L2-normalized spectra,Direct logistic regression (L2),0.8052 ± 0.0278,0.8685 ± 0.0306,0.9349 ± 0.0135
2,Direct L2-normalized spectra,Direct weighted kNN + standardized Euclidean,0.6822 ± 0.0331,0.7391 ± 0.0391,0.8771 ± 0.0186
3,FPLS scores,FPLS + linear SVM (J=5),0.8082 ± 0.0295,0.8658 ± 0.0309,0.9305 ± 0.0149
4,FPLS scores,FPLS + logistic regression (J=5),0.8070 ± 0.0289,0.8698 ± 0.0301,0.9319 ± 0.0145
5,FPLS scores,FPLS + weighted kNN (J=5),0.8001 ± 0.0300,0.8220 ± 0.0307,0.8974 ± 0.0164


,comparison_group,method_a,method_b,metric,n_pairs,mean_a,mean_b,mean_diff_a_minus_b,wilcoxon_stat,p_value,significant_0_05
0,A. FPLS model comparison,FPLS + linear SVM (J=5),FPLS + logistic regression (J=5),f1,50,0.808243,0.807004,0.001239,606.0,7.667327e-01,False
1,A. FPLS model comparison,FPLS + linear SVM (J=5),FPLS + logistic regression (J=5),pr_auc,50,0.865758,0.869791,-0.004032,3.0,8.881784e-15,True
2,A. FPLS model comparison,FPLS + linear SVM (J=5),FPLS + weighted kNN (J=5),f1,50,0.808243,0.800105,0.008138,434.0,4.944641e-02,True
3,A. FPLS model comparison,FPLS + linear SVM (J=5),FPLS + weighted kNN (J=5),pr_auc,50,0.865758,0.821988,0.043770,0.0,1.776357e-15,True
4,A. FPLS model comparison,FPLS + logistic regression (J=5),FPLS + weighted kNN (J=5),f1,50,0.807004,0.800105,0.006899,481.0,1.329606e-01,False
5,A. FPLS model comparison,FPLS + logistic regression (J=5),FPLS + weighted kNN (J=5),pr_auc,50,0.869791,0.821988,0.047802,0.0,1.776357e-15,True
6,B. Direct spectra vs FPLS,Direct linear SVM,FPLS + linear SVM (J=5),f1,50,0.812075,0.808243,0.003832,531.0,3.090510e-01,False
7,B. Direct spectra vs FPLS,Direct linear SVM,FPLS + linear SVM (J=5),pr_auc,50,0.866675,0.865758,0.000917,524.0,2.779880e-01,False
8,B. Direct spectra vs FPLS,Direct logistic regression (L2),FPLS + logistic regression (J=5),f1,50,0.805179,0.807004,-0.001825,579.0,7.389574e-01,False
9,B. Direct spectra vs FPLS,Direct logistic regression (L2),FPLS + logistic regression (J=5),pr_auc,50,0.868510,0.869791,-0.001281,540.0,3.521750e-01,False
